In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
import cv2
import gymnasium as gym
import numpy as np

from collections import deque

#wrapper for env. will crop to 86x86 like in the paper and stack 4 frames, returning a (4, 86, 86)
class BuildState(gym.Wrapper):

    def __init__(self, env, k=4, training=True):
        super().__init__(env)
        self.k = k
        self.frames = deque([],maxlen=4)
        self.was_real_terminated = True
        self.training = training

        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(4, 84, 84),
            dtype=np.uint8 #changed from 32, was too big for buffer
        )

    # convert to greyscale and crop img to a 84x84
    def process_image(self,img):
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        resized_img = cv2.resize(gray_img, (84, 110), interpolation=cv2.INTER_AREA)

        # remove from the top, the score board
        cropped_img  = resized_img[18:102, 0:84]
        # normalize for nn
        return cropped_img.astype(np.uint8)


    #start a new game. get inital img, crop and copy it 4 times to fit nn tmplt
    def reset(self,**kwargs):
        # Only completely restart the emulator if all lives are gone
        if self.was_real_terminated or not self.training:
            obs, info = self.env.reset(**kwargs)
        else:
            # We just lost a ball. Don't reset! Just step once to get the frame.
            obs, _, _, _, info = self.env.step(0)

        # Force fire to launch the ball (This now works perfectly for both New Games AND New Lives)
        # obs, _, _, _, info = self.env.step(1)
        self.lives = info.get('lives', 5)

        p_obs = self.process_image(obs)
        for _ in range(self.k):
            self.frames.append(p_obs)

        return np.stack(self.frames, axis=0), info


    '''
    overwrites the step function. will return 4 stack of greysacle 84x84 images
    NOTICE- will aplly the SAME action to all k frames. the reward
    TODO: its unclear how the reward works. sum it up?
    '''
    def step(self, action):
        tottal_reward = 0.0
        for _ in range(self.k):
            observation, reward, terminated, truncated, info = self.env.step(action)
            self.was_real_terminated = terminated

            current_lives = info.get('lives', 5)

            # ONLY fake the termination if we are in training mode
            if self.training and current_lives < self.lives:
                terminated = True

            self.lives = current_lives
            tottal_reward += reward

            if terminated or truncated:
                break

        p_obs = self.process_image(observation)
        self.frames.append(p_obs)
        return np.stack(self.frames, axis=0), tottal_reward, terminated, truncated, info



In [ ]:
import random
from collections import deque, namedtuple

Frame = namedtuple('frame',('state','action','reward','next_state', 'ended'))

class Replay_buffer:
    def __init__(self,capacity):
        self.capacity = capacity
        # Pre-allocate memory as single 84x84 arrays (8x smaller footprint)
        self.frames = np.zeros((capacity, 84, 84), dtype=np.uint8)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.dones = np.zeros(capacity, dtype=bool)

        self.ptr = 0
        self.size = 0

    #add new fram to buffer
    def push(self, frame, action, reward, done):
        self.frames[self.ptr] = frame
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.dones[self.ptr] = done
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)
    
#TODO - unerstand...
    def _get_stack(self, idx):
        stack = np.zeros((4, 84, 84), dtype=np.uint8)
        curr_idx = idx
        for i in range(3, -1, -1):
            stack[i] = self.frames[curr_idx]
            prev_idx = (curr_idx - 1) % self.capacity
            # Stop looking backward if game ended, pad remaining with current frame
            if self.dones[prev_idx] or prev_idx == self.ptr:
                for j in range(i - 1, -1, -1):
                    stack[j] = stack[i]
                break
            curr_idx = prev_idx
        return stack
    
    #sample X frames from buffer
    def sample(self, batch_size):
        valid_indices = []
        while len(valid_indices) < batch_size:
            idx = np.random.randint(0, self.size)
            if (idx + 1) % self.capacity == self.ptr:
                continue
            if self.size < self.capacity and idx < 3:
                continue
            valid_indices.append(idx)

        states = np.zeros((batch_size, 4, 84, 84), dtype=np.uint8)
        next_states = np.zeros((batch_size, 4, 84, 84), dtype=np.uint8)
        actions = np.zeros(batch_size, dtype=np.int64)
        rewards = np.zeros(batch_size, dtype=np.float32)
        dones = np.zeros(batch_size, dtype=np.float32)

        for i, idx in enumerate(valid_indices):
            states[i] = self._get_stack(idx)
            next_states[i] = self._get_stack((idx + 1) % self.capacity)
            actions[i] = self.actions[idx]
            rewards[i] = self.rewards[idx]
            dones[i] = self.dones[idx]

        return states, actions, rewards, next_states, dones

    #allows len to work on this class in other files
    def len(self):
        return self.size


In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as opt
import numpy as np



class DNQ(nn.Module):
    def __init__(self, output_size=9):
        super(DNQ, self).__init__()
        #conv 4,84,84-> 16, 20,20
        self.conv1 = nn.Conv2d(4,16,kernel_size=8,stride=4)
        #conv 16,20,20-> 32,9,9

        self.conv2 = nn.Conv2d(16,32,kernel_size=4,stride=2)

        self.fc1 = nn.Linear(32 * 9 * 9 ,256)
        self.fc2 = nn.Linear(256 ,output_size)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        #flatten to fit the fc
        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))

        return (self.fc2(x))



class DNQAgent:
    def __init__(self,  device, actions=9, gamma=0.99):

        #setup DNQ
        self.device = device
        self.DNQ = DNQ(actions).to(self.device)
        self.loss_fn = torch.nn.SmoothL1Loss() # gemini is saying SmoothL1Loss. test that and mse
        # GEMINI thinks i need to use a nigger aplph. will try, not sure what hes absing it on
        self.optimizer= opt.RMSprop(self.DNQ.parameters(), lr=0.0005, alpha=0.95, eps=0.01) #well play with the lr later .Adam or msprop?
        self.gamma = gamma


    # infrence time, finds best action
    def select_action(self, frame):
        state_tensor = torch.from_numpy(np.array(frame)).float().unsqueeze(0).to(self.device) / 255.0
        self.DNQ.eval()
        with torch.no_grad():
            # need to add 'unsqueeze' for cnn to work, adds a dim in satart for batch size
            q_values = self.DNQ(state_tensor)
        self.DNQ.train()

        #pick the acton with highst score
        best_action = q_values.argmax().detach().cpu().item()
        return best_action


    '''
        here is the entire learning process. recives a batch of state from the
        buffer {('state','action','reward','next_state', 'ended'),()...}. ff to find *current* reward for action Si
        then plugges into bellmon eq then fowrd on Si+1 and MSE on the dif
    '''
    def learn_samples(self, batch):
        #extract state, conter to tensor. size: (batch_size,4,84,84). ff-> (batch_size,9) (every action per state)

        states, actions, rewards, next_states, dones = batch
        #convert to tensors
        states_t = torch.tensor(np.array(states), dtype=torch.float32).to(self.device) / 255.0
        actions_t = torch.tensor(actions, dtype=torch.int64).unsqueeze(1).to(self.device)
        rewards_t = torch.tensor(rewards, dtype=torch.float32).to(self.device)

        # bounding transformation [-1.0, 1.0]
        rewards_t = torch.clamp(rewards_t, min=-1.0, max=1.0).to(self.device)

        next_states_t = torch.tensor(np.array(next_states), dtype=torch.float32).to(self.device) / 255.0
        dones_t = torch.tensor(dones, dtype=torch.float32).to(self.device)

       # set up the Qs+1 values for bellmon, meaning how much we'd make from next_state
        with torch.no_grad():

            next_q_values = self.DNQ(next_states_t)
            #pick the acton with highst score GEMINI said to add .detach(). see if works#########################
            max_next_q_values = next_q_values.max(1)[0].detach()
            #trick from gemini,if finished it will be only the reward, like the paper
            expected_q_values = rewards_t + (self.gamma * max_next_q_values * (1 - dones_t))


        #pick the matching action to what was done, set as
        q_values = self.DNQ(states_t)

        current_q_values = q_values.gather(1, actions_t).squeeze(1)

        loss = self.loss_fn(current_q_values, expected_q_values)

        self.optimizer.zero_grad()
        loss.backward()

        # ==========================================
        # DIAGNOSTIC: Calculate the L2 Gradient Norm
        # ==========================================
        #total_norm = 0.0
        #for p in self.DNQ.parameters():
        #    if p.grad is not None:
        #        # Calculate the L2 norm of the gradients for this specific layer
        #        param_norm = p.grad.data.norm(2)
        #        # Square it and add to the total sum
        #        total_norm += param_norm.item() ** 2
        #
        ## Take the square root of the total sum
        #total_norm = total_norm ** 0.5
        #
        ## Print the scalar loss and the magnitude of the update
        #print(f"Loss: {loss.item():.4f} | Update Magnitude (L2 Norm): {total_norm:.4f}")
        # ==========================================
        torch.nn.utils.clip_grad_norm_(self.DNQ.parameters(), max_norm=1.0) # GEMINI IDEA> TEST FOR DIF
        self.optimizer.step()








In [ ]:
#from DNQ_agent import DNQAgent
#from replay_buffer import Replay_buffer
#from build_state import BuildState
####### FOR COLAB
import gc
import ctypes
##########


import matplotlib.pyplot as plt
import gymnasium as gym
import ale_py
import random
import torch
import numpy as np
import os

BUFFER_SIZE = 1000000 # 400,000 kangaro / pinball 12 / Amidar -  should be tenth of tottal frames. should start seeing mprovment faster then what i do...
EPSILON = 0.11
BATCH_SIZE=32 #bigger batch - 64
REVIEW_FREQUENCY =10000 #steps
MAX_STEPS = 400000 #2500000 ?
MIN_BUFFER_SIZE = 1000
goal_score = 230 
def run_eval_episodes(env, agent, device, num_episodes=3):
    all_scores = []

    # Temporarily disable fake termination for evaluation
    # We use getattr to safely check if the wrapper is being used
    original_training_mode = getattr(env, 'training', True)
    env.training = False

    for episode in range(num_episodes):
        state, _ = env.reset()
        terminated = False
        truncated = False
        episode_score = 0

        while not (truncated or terminated):
            state_tensor = torch.from_numpy(np.array(state)).float().unsqueeze(0).to(device) / 255.0
            with torch.no_grad():
                q_values = agent.DNQ(state_tensor)
                best_action = q_values.argmax().item()

            state, reward, terminated, truncated, _ = env.step(best_action)
            episode_score += reward
            #print(best_action)###DEBUG

        all_scores.append(episode_score)

    # Restore the original mode (important if you continue training after eval)
    env.training = original_training_mode

    avg_eval_score = sum(all_scores) / num_episodes
    return avg_eval_score


def plot_training_results(score_history, review_freq=10000):
    fig, ax = plt.subplots(figsize=(10, 5))

    # Calculate actual steps for the X-axis
    steps = [i * review_freq for i in range(1, len(score_history) + 1)]

    # --- Plot Scores ---
    color = 'tab:blue'
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Score per Evaluation', color=color)
    
    # Plot the raw scores as a lighter line
    ax.plot(steps, score_history, color=color, alpha=0.4, label='Raw Score')

    # Add a moving average for the score to see the trend through the noise
    if len(score_history) >= 10:
        moving_avg = np.convolve(score_history, np.ones(10)/10, mode='valid')
        # Shift the X-axis for the moving average so it aligns correctly
        ma_steps = steps[9:] 
        ax.plot(ma_steps, moving_avg, color='red', linewidth=2, label='Score Trend (MA10)')

    fig.tight_layout()
    plt.title("2013 DQN Training Progress: amidar")
    plt.legend()
    
    # THE CRITICAL LINES FOR THE OVERNIGHT RUN
    plt.savefig("/kaggle/working/training_progress.png")
    plt.close()



def main():
    #setup
    gym.register_envs(ale_py)
    basic_env = gym.make("AmidarNoFrameskip-v4")
    # "RiverraidNoFrameskip-v4"  , render_mode="human",VideoPinballNoFrameskip-v4 AmidarNoFrameskip-v4

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    #load_path = '/kaggle/input/datasets/etlbro/amidar-r3/Amidar_v2_r3.pth'
    
    env = BuildState(basic_env,k=4) #k - frame skips
    num_actions = env.action_space.n
    agent = DNQAgent(device=device, actions=num_actions)
    
    #agent.DNQ.load_state_dict(torch.load(load_path, map_location=device, weights_only=True))
    #print("Weights loaded successfully!")
    #agent.DNQ.train()
    
    # 1. Define the path to the file you want to load
    load_path = '/kaggle/input/datasets/etlbro/amidar-f-v4/Amidar_Latest_v4.pth'
   ##
    if os.path.exists(load_path):
        print(f"Loading weights from {load_path}...")
        agent.DNQ.load_state_dict(torch.load(load_path, map_location=device, weights_only=True))
        print("Weights loaded successfully.")
   # else:
   #     print(f"No checkpoint found at {load_path}. Starting from scratch.")

    buffer = Replay_buffer(capacity=BUFFER_SIZE)

    #evaluation vars-
    tottal_score = 0
    eval_frames = None
    avgQ_history = []
    avg_score_history = []
    step_count = 0
    epsilon = EPSILON
    episode = 0
    best_score = 120
    
    while step_count < MAX_STEPS: #while True:
        #episodes (games-rounds) loop. starts new game

        new_frame,_ = env.reset()
        terminated = False
        truncated = False
        episode_reward = 0
        episode_number = 0
        ended = 0

        while not ended:
            step_count +=1
            #note- this env is a wrapper, the frames are already stacks of k frames
            old_frame = new_frame
            
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = agent.select_action(old_frame)
            
            new_frame, reward, terminated, truncated, info = env.step(action)


            ended = terminated or truncated
            #save to buffer
            buffer.push(new_frame[-1], action, reward, ended)
            '''         getting rid of q eval. not helping or acurate   
            #builf Q evel when we have enought frames
            if eval_frames is None and buffer.len() > 10000:

                states_only, _, _, _, _ = buffer.sample(500)
                
                eval_frames = torch.tensor(np.array(states_only), dtype=torch.float32).to(device) / 255.0
                print("set evaluation set of 500 states captured.")
            '''
            if epsilon > 0.1: # first 900,000 eps shrinks
                epsilon -= 0.000001

            #now select from buffer and do the learning part
            if buffer.len() > MIN_BUFFER_SIZE and step_count % 4 == 0:
                batch = buffer.sample(BATCH_SIZE)# = batch size
                agent.learn_samples(batch)  

            if step_count % REVIEW_FREQUENCY == 0:
                print(f"\n--- EVALUATION AT STEP {step_count} ---")

                # 1. Run the evaluation episodes
                # We use basic_env or a separate eval_env to avoid wrapper conflicts
                avg_eval_reward = run_eval_episodes(env, agent, device, num_episodes=3)
                avg_score_history.append(avg_eval_reward)
               
                # 2. Q-Value Avg Eval (Your existing logic)
                '''               if eval_frames is not None:
                    with torch.no_grad():
                        all_q_values = agent.DNQ(eval_frames)
                        max_q_values = all_q_values.max(1)[0]
                        avg_q = max_q_values.mean().item()
                        avgQ_history.append(avg_q)
                else:
                    avg_q = 0.0
                '''
                print(f"Step: {step_count} | Eval Reward: {avg_eval_reward} | epsilom: {epsilon}") #| Avg Q: {avg_q:.4f} 

                #show_agent_play(agent, device)

                     # --- AUTO-SAVE CHECKPOINT ---
              # We use a f-string to include the episode number in the filename
               # checkpoint_path = f'/content/drive/MyDrive/dl/RL/assin1/boxing_v1.pth'

               # torch.save(agent.DNQ.state_dict(), checkpoint_path)
                torch.save(agent.DNQ.state_dict(), "/kaggle/working/Amidar_Latest.pth")
                #kaggle version
                if avg_eval_reward > best_score:
                    best_score = avg_eval_reward
                    print(f"---NEW BEST SCORE: {best_score}----")
                    torch.save(agent.DNQ.state_dict(), "/kaggle/working/Amidar_v4.pth")
                    
            # if i reach goal score, no reson to continue....
            if best_score >= goal_score:
                   step_count = MAX_STEPS
    if len(avg_score_history) > 0:
        total_average = sum(avg_score_history) / len(avg_score_history)
        print(f"Overall Average Score: {total_average:.2f}")
    env.close()

    plot_training_results(avg_score_history, review_freq=REVIEW_FREQUENCY)

    # observation,_ = env.reset()
    # action = env.action_space.sample()
    # observation, reward, terminated, truncated, _= env.step(action)






if __name__ == '__main__':
    main()











In [ ]:
from IPython.display import FileLink
FileLink(r'Amidar_v4.pth')

In [ ]:
from IPython.display import Image, FileLink, display
import os

# The exact path where your code saved the graph
file_path = "/kaggle/working/training_progress.png"

if os.path.exists(file_path):
    # 1. Show the image right here in the notebook
    print("Here is your final 1-Million-Step training graph!")
    display(Image(filename=file_path))

    # 2. Generate the clickable download link
    print("\nClick the link below to download it for your PDF report:")
    display(FileLink(file_path))
else:
    print("Error: Could not find the graph file. Make sure the run fully finished and didn't crash!")

In [ ]:
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
from IPython.display import Image, FileLink, display
import os

def load_and_evaluate(load_path, num_episodes=10, eval_epsilon=0.05):
    print(f"Loading environment and weights from: {load_path}")
    
    # 1. Setup Environment and Agent
    basic_env = gym.make("AmidarNoFrameskip-v4")
    env = BuildState(basic_env, k=4)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = DNQAgent(device=device, actions=env.action_space.n)
    
    # 2. Load the Brain
    try:
        agent.DNQ.load_state_dict(torch.load(load_path, map_location=device, weights_only=True))
        print("Weights loaded successfully!")
    except Exception as e:
        print(f"Error loading weights! Check your file path. Error: {e}")
        return

    # Lock the weights!
    agent.DNQ.eval()
    
    # Turn off your custom training wrapper logic
    original_training_mode = getattr(env, 'training', True)
    env.training = False

    episode_scores = []

    # 3. Play X Episodes
    print(f"\nStarting Evaluation for {num_episodes} episodes with {eval_epsilon*100}% randomness...")
    
    for episode in range(1, num_episodes + 1):
        state, _ = env.reset()
        terminated = False
        truncated = False
        episode_score = 0

        while not (truncated or terminated):
            # 5% Randomness to prevent infinite loops (2013 Paper Rule)
            if random.random() < eval_epsilon:
                best_action = env.action_space.sample()
            else:
                state_tensor = torch.from_numpy(np.array(state)).float().unsqueeze(0).to(device) / 255.0
                with torch.no_grad():
                    q_values = agent.DNQ(state_tensor)
                    best_action = q_values.argmax().item()

            state, reward, terminated, truncated, _ = env.step(best_action)
            episode_score += reward

        print(f"Episode {episode} | Score: {episode_score}")
        episode_scores.append(episode_score)

    # Clean up
    env.training = original_training_mode
    env.close()

    # 4. Calculate Final Stats
    total_score = sum(episode_scores)          # <--- ADDED THIS
    avg_score = total_score / num_episodes     # <--- Updated to use the total
    max_score = max(episode_scores)
    min_score = min(episode_scores)
    
    print(f"\n--- EVALUATION COMPLETE ---")
    print(f"Total Score (All Rounds): {total_score}") # <--- ADDED THIS
    print(f"Average Score: {avg_score}")
    print(f"Highest Score: {max_score}")
    print(f"Lowest Score:  {min_score}")

    # 5. Draw the Graph
    plt.figure(figsize=(10, 5))
    
    # Plotting individual games as a bar chart makes it easy to see the variance
    bars = plt.bar(range(1, num_episodes + 1), episode_scores, color='skyblue', edgecolor='black')
    
    # Draw a bold line showing the mathematical average
    plt.axhline(y=avg_score, color='red', linestyle='--', linewidth=2, label=f'Average ({avg_score:.1f})')
    
    plt.title(f'Final Model Evaluation ({num_episodes} Episodes)')
    plt.xlabel('Episode Number')
    plt.ylabel('Score')
    plt.xticks(range(1, num_episodes + 1))
    plt.legend()
    
    # Add the exact scores on top of each bar for readability
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 10, int(yval), ha='center', va='bottom', fontsize=9)

    # 6. Save and Show
    save_path = "/kaggle/working/final_evaluation_graph.png"
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()
    plt.close()

    # 7. Generate Download Link
    print("\nGraph saved successfully! Click below to download for your report:")
    display(FileLink(save_path))


# ==========================================
# RUN THE SCRIPT HERE
# ==========================================

# Make sure this path points exactly to the file you want to test!
# Example: /kaggle/working/Riverraid_V3.pth OR /kaggle/input/...
MY_MODEL_PATH = '/kaggle/working/Amidar_v4.pth' 

# Test it across 10 games (you can change this to 100 for the official paper standard!)
load_and_evaluate(load_path=MY_MODEL_PATH, num_episodes=50, eval_epsilon=0.05)